# Google Colab Host for EduMind AI Backend & Ollama

Run this single cell to clone the repo, mount Google Drive to pull source documents, automatically rebuild the vector store using GPU, install/start local Ollama, pull the LLM model, configure `.env` to point to localhost Ollama, and host the FastAPI backend server publicly using Cloudflare Tunnel.

In [ ]:
# Force working directory to Colab's default /content directory to prevent nested cloning
import os
os.chdir('/content')

# =====================================================================
# 1. Clone the GitHub Repository & Switch to Feature Branch
# =====================================================================
import getpass
import shutil
import zipfile
import subprocess
import time
import re

repo_url = "github.com/jaynishthakar/demo-.git"
project_dir = "demo-"
branch_name = "feature/colab-tunnel-setup"

if not os.path.exists(project_dir):
    print("Cloning repository...")
    is_private = input("Is the GitHub repository private? (yes/no): ").strip().lower() == "yes"
    if is_private:
        token = getpass.getpass("Enter your GitHub Personal Access Token (PAT): ").strip()
        clone_url = f"https://{token}@{repo_url}"
    else:
        clone_url = f"https://{repo_url}"

    !git clone -b {branch_name} {clone_url}
else:
    print("Repository already exists. Pulling latest updates...")
    # Clean up any nested directories if they exist from previous runs
    nested_path = os.path.join(project_dir, project_dir)
    if os.path.exists(nested_path):
        shutil.rmtree(nested_path)
        
    %cd {project_dir}
    !git checkout {branch_name}
    !git pull
    %cd ..

%cd {project_dir}

# =====================================================================
# 2. Install Python & System Dependencies
# =====================================================================
print("\nInstalling system dependencies (zstd)...")
!apt-get update && apt-get install -y zstd

print("\nInstalling python packages (requirements.txt)...")
!pip install -r requirements.txt

# =====================================================================
# 3. Mount Google Drive, Restore Docs, & Rebuild Vector Store
# =====================================================================
try:
    print("\nAttempting to mount Google Drive to check for source documents...")
    from google.colab import drive
    drive.mount('/content/drive')
    
    zip_path = "/content/drive/MyDrive/VIT Final SOPs-01-07-2012.zip"
    if os.path.exists(zip_path):
        print("Extracting source documents directly from Google Drive...")
        # Clean up any old data directories to prevent mismatches
        if os.path.exists("data"):
            shutil.rmtree("data")
        if os.path.exists("vector_store/chroma_db"):
            shutil.rmtree("vector_store/chroma_db")
            
        staging_dir = "data/staging"
        os.makedirs(staging_dir, exist_ok=True)
        
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            temp_extract = "temp_docs"
            zip_ref.extractall(temp_extract)
            
            sub_dir = os.path.join(temp_extract, "VIT Final SOPs-01-07-2012")
            src_folder = sub_dir if os.path.exists(sub_dir) else temp_extract
            for file_name in os.listdir(src_folder):
                src_file = os.path.join(src_folder, file_name)
                dest_file = os.path.join(staging_dir, file_name)
                if os.path.isfile(src_file):
                    shutil.copy2(src_file, dest_file)
            
            shutil.rmtree(temp_extract)
            
        print("SUCCESS: Source documents staged in data/staging/.")
        
        # Run ingestion and indexing using shell commands so output prints in real-time
        print("\nBuilding SQLite database and generating GPU vector embeddings in ChromaDB...")
        !python ingestion_pipeline.py
        !python vector_store/index_pipeline.py
        print("SUCCESS: Vector store and metadata database are fully rebuilt!")
    else:
        print(f"\n[NOTE] 'VIT Final SOPs-01-07-2012.zip' not found in your Drive at: {zip_path}")
        print("Please place the ZIP file in your main Google Drive folder to automatically restore source documents.")
except Exception as e:
    print(f"\n[INFO] Google Drive mounting or auto-ingestion skipped/failed: {e}")
    print("Continuing backend startup without pre-ingested source documents.")

# =====================================================================
# 4. Install & Start Ollama (CORS Enabled)
# =====================================================================
print("\nInstalling Ollama...")
!curl -fsSL https://ollama.com/install.sh | sh

print("\nStarting Ollama service in background...")
env = os.environ.copy()
env["OLLAMA_ORIGINS"] = "*"
subprocess.Popen(["ollama", "serve"], env=env, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)

print("\nPulling Qwen2.5:7b model (takes ~1-2 mins)...")
!ollama pull qwen2.5:7b

# =====================================================================
# 5. Start the FastAPI Backend
# =====================================================================
print("\nStarting FastAPI Backend in the background...")
backend_env = os.environ.copy()
backend_env["OLLAMA_BASE_URL"] = "http://localhost:11434"
backend_env["LLM_BACKEND"] = "ollama"
backend_env["OLLAMA_MODEL"] = "qwen2.5:7b"

backend_log = open("backend_server.log", "w")
backend_process = subprocess.Popen(
    ["python", "-m", "uvicorn", "backend.app:app", "--host", "0.0.0.0", "--port", "8000"],
    env=backend_env, stdout=backend_log, stderr=backend_log
)
time.sleep(8)

# =====================================================================
# 6. Install & Start Cloudflare Tunnel on Backend Port (8000)
# =====================================================================
if not os.path.exists("cloudflared-linux-amd64.deb"):
    print("\nDownloading Cloudflare Tunnel client...")
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
    !dpkg -i cloudflared-linux-amd64.deb

print("\nStarting Cloudflare Tunnel on Backend Port 8000...")
max_attempts = 5
for attempt in range(1, max_attempts + 1):
    print(f"  Starting tunnel attempt {attempt}/{max_attempts}...")
    !pkill cloudflared
    time.sleep(2)
    
    with open("cloudflare_tunnel.log", "w") as log_file:
        tunnel_process = subprocess.Popen([
            "cloudflared", "tunnel", "--protocol", "http2",
            "--url", "http://localhost:8000"
        ], stdout=log_file, stderr=log_file)
        
    time.sleep(10)
    
    with open("cloudflare_tunnel.log", "r") as log_file:
        log_content = log_file.read()
        urls = re.findall(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', log_content)
        if urls:
            print("\n" + "="*70)
            print(" SUCCESS: BACKEND IS NOW ONLINE VIA CLOUDFLARE")
            print("-"*70)
            print(" Copy this URL into your Vercel Frontend UI:")
            print(f" {urls[0]}")
            print("="*70 + "\n")
            break
        else:
            print(f"  [WARNING] Tunnel attempt {attempt} did not establish URL yet.")
            if attempt == max_attempts:
                print("\n[ERROR] Failed to establish Cloudflare Tunnel. Printing logs:")
                print(log_content)
            else:
                print("  Retrying in 5 seconds...")
                time.sleep(5)

# Keep cell alive
try:
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    print("\nStopping tunnel, backend, and Ollama...")
finally:
    !pkill cloudflared
    backend_process.terminate()
    print("Cleanup complete.")
